In [45]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.ansi.enabled", "false")


def transform_bronze_rentals(bronze_df):
    """
    Produce the valid Silver rentals and quarantine tables
    deterministically from Bronze.
    """

    rental_window = (
        Window
        .partitionBy("rental_id")
        .orderBy(
            F.col("ingested_at").desc(),
            F.col("source_file").desc()
        )
    )

    # Keep exactly one source row per rental_id.
    rentals_deduped = (
        bronze_df
        .withColumn(
            "_row_number",
            F.row_number().over(rental_window)
        )
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )

    rental_type_token = F.upper(
        F.regexp_replace(
            F.trim(F.col("rental_type")),
            r"[^A-Za-z]",
            ""
        )
    )

    rentals_pre_quality = (
        rentals_deduped
        .withColumn(
            "checkout_ts",
            F.to_timestamp(
                F.col("checkout_ts"),
                "yyyy-MM-dd HH:mm:ss"
            )
        )
        .withColumn(
            "checkin_ts",
            F.to_timestamp(
                F.when(
                    F.trim(F.col("checkin_ts")) == "",
                    F.lit(None)
                ).otherwise(F.col("checkin_ts")),
                "yyyy-MM-dd HH:mm:ss"
            )
        )
        .withColumn(
            "rental_type",
            F.when(
                rental_type_token.rlike(
                    r"^(STANDARD|STD)"
                ),
                F.lit("standard")
            )
            .when(
                rental_type_token.rlike(
                    r"^(PRIORITY|PRIO|PRI)"
                ),
                F.lit("priority")
            )
        )
    )

    # Impossible only when a real check-in is before checkout.
    bad = (
        F.col("checkin_ts")
        < F.col("checkout_ts")
    )

    # NULL check-in is kept because the rental is still active.
    valid_rentals = (
        rentals_pre_quality
        .filter(
            ~F.coalesce(
                bad,
                F.lit(False)
            )
        )
    )

    quarantined_rentals = (
        rentals_pre_quality
        .filter(
            F.coalesce(
                bad,
                F.lit(False)
            )
        )
        .withColumn(
            "quarantine_reason",
            F.lit("checkin_before_checkout")
        )
    )

    return valid_rentals, quarantined_rentals

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 47, Finished, Available, Finished, False)

In [46]:
def replace_delta_table(dataframe, table_name):
    """
    Replace the table contents instead of appending.
    """

    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )


def build_silver_rentals():
    """
    Rebuild both Silver rental outputs from the full
    Bronze rental snapshot.
    """

    bronze_rentals = spark.table("bronze_rentals")

    valid_rentals, quarantined_rentals = (
        transform_bronze_rentals(bronze_rentals)
    )

    valid_count = valid_rentals.count()
    quarantine_count = quarantined_rentals.count()

    duplicate_valid_ids = (
        valid_rentals
        .groupBy("rental_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    overlapping_ids = (
        valid_rentals
        .select("rental_id")
        .join(
            quarantined_rentals.select("rental_id"),
            "rental_id",
            "inner"
        )
        .count()
    )

    assert duplicate_valid_ids == 0, (
        "Duplicate rental IDs remain in Silver."
    )

    assert overlapping_ids == 0, (
        "A rental ID appears in both Silver and quarantine."
    )

    replace_delta_table(
        valid_rentals,
        "silver_rentals"
    )

    replace_delta_table(
        quarantined_rentals,
        "silver_rentals_quarantine"
    )

    result = {
        "silver_rentals": valid_count,
        "silver_rentals_quarantine": quarantine_count
    }

    print(result)

    return result

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 48, Finished, Available, Finished, False)

In [47]:
first_run = build_silver_rentals()

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 49, Finished, Available, Finished, False)

{'silver_rentals': 947, 'silver_rentals_quarantine': 3}


In [48]:
second_run = build_silver_rentals()

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 50, Finished, Available, Finished, False)

{'silver_rentals': 947, 'silver_rentals_quarantine': 3}


In [49]:
print("First run: ", first_run)
print("Second run:", second_run)

assert first_run == second_run

assert first_run["silver_rentals"] == 947
assert first_run["silver_rentals_quarantine"] == 3

print(
    "Idempotency passed: running the build twice "
    "produced identical counts."
)

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 51, Finished, Available, Finished, False)

First run:  {'silver_rentals': 947, 'silver_rentals_quarantine': 3}
Second run: {'silver_rentals': 947, 'silver_rentals_quarantine': 3}
Idempotency passed: running the build twice produced identical counts.


In [50]:
quarantined_ids = (
    spark.table("silver_rentals_quarantine")
    .select("rental_id")
)

quarantined_ids_in_valid_table = (
    spark.table("silver_rentals")
    .join(
        quarantined_ids,
        "rental_id",
        "inner"
    )
    .count()
)

print(
    "Quarantined IDs found in silver_rentals:",
    quarantined_ids_in_valid_table
)

assert quarantined_ids_in_valid_table == 0

print(
    "No quarantined rental was brought back "
    "into silver_rentals."
)

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 52, Finished, Available, Finished, False)

Quarantined IDs found in silver_rentals: 0
No quarantined rental was brought back into silver_rentals.


In [51]:
duplicate_ids = (
    spark.table("silver_rentals")
    .groupBy("rental_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_ids == 0

print("silver_rentals contains one row per rental_id.")

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 53, Finished, Available, Finished, False)

silver_rentals contains one row per rental_id.


## 5. Add the new daily rental file

#### Create a test daily file with five valid rentals

In [52]:
import csv
from pathlib import Path

TEST_FILE_NAME = "rentals_2026-07-02.csv"

TEST_FILE_PATH = Path(
    "/lakehouse/default/Files/"
    "buildmate_raw_data/raw/rentals/"
    f"{TEST_FILE_NAME}"
)

# Use existing valid customer, depot, asset, and rental-type values
# so the test records remain consistent with the source data.
sample_rows = (
    spark.table("silver_rentals")
    .select(
        "customer_id",
        "depot_code",
        "asset_id",
        "rental_type"
    )
    .limit(5)
    .collect()
)

assert len(sample_rows) == 5

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 54, Finished, Available, Finished, False)

#### Generate five unused rental IDs:

In [53]:
existing_rental_ids = {
    row["rental_id"]
    for row in (
        spark.table("silver_rentals")
        .select("rental_id")
        .union(
            spark.table("silver_rentals_quarantine")
            .select("rental_id")
        )
        .distinct()
        .collect()
    )
}

new_rental_ids = []
candidate_number = 990001

while len(new_rental_ids) < 5:
    candidate_id = f"R{candidate_number}"

    if candidate_id not in existing_rental_ids:
        new_rental_ids.append(candidate_id)

    candidate_number += 1

print("New test rental IDs:", new_rental_ids)

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 55, Finished, Available, Finished, False)

New test rental IDs: ['R990001', 'R990002', 'R990003', 'R990004', 'R990005']


#### Write the raw CSV:

In [54]:
TEST_FILE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

with TEST_FILE_PATH.open(
    mode="w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.writer(file)

    writer.writerow([
        "rental_id",
        "customer_id",
        "depot_code",
        "asset_id",
        "checkout_ts",
        "checkin_ts",
        "rental_type"
    ])

    for index, row in enumerate(sample_rows):
        checkout_hour = 8 + index
        checkin_hour = 8 + index

        writer.writerow([
            new_rental_ids[index],
            row["customer_id"],
            row["depot_code"],
            row["asset_id"],
            f"2026-07-01 {checkout_hour:02d}:00:00",
            f"2026-07-02 {checkin_hour:02d}:00:00",
            row["rental_type"].upper()
        ])

print(f"Created test file: {TEST_FILE_PATH}")

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 56, Finished, Available, Finished, False)

Created test file: /lakehouse/default/Files/buildmate_raw_data/raw/rentals/rentals_2026-07-02.csv


#### 6. Record the existing Silver state

In [55]:
before_silver_count = (
    spark.table("silver_rentals").count()
)

before_quarantine_count = (
    spark.table("silver_rentals_quarantine").count()
)

known_rental_ids = (
    spark.table("silver_rentals")
    .select("rental_id")
    .union(
        spark.table("silver_rentals_quarantine")
        .select("rental_id")
    )
    .distinct()
)

print(
    f"Silver rentals before new file: "
    f"{before_silver_count}"
)

print(
    f"Quarantine before new file: "
    f"{before_quarantine_count}"
)

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 57, Finished, Available, Finished, False)

Silver rentals before new file: 947
Quarantine before new file: 3


#### 7. Refresh Bronze from the wildcard

In [56]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

RAW = "Files/buildmate_raw_data/raw"

rentals_schema = StructType([
    StructField("rental_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("depot_code", StringType(), True),
    StructField("asset_id", StringType(), True),
    StructField("checkout_ts", StringType(), True),
    StructField("checkin_ts", StringType(), True),
    StructField("rental_type", StringType(), True),
])

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 58, Finished, Available, Finished, False)

In [57]:
bronze_rentals_refreshed = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(rentals_schema)
    .load(
        f"{RAW}/rentals/rentals_*.csv"
    )
    .withColumn(
        "ingested_at",
        F.current_timestamp()
    )
    .withColumn(
        "source_file",
        F.input_file_name()
    )
)

(
    bronze_rentals_refreshed.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_rentals")
)

print(
    "Bronze rentals refreshed from all files."
)

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 59, Finished, Available, Finished, False)

Bronze rentals refreshed from all files.


#### 8. Count the new day’s valid rentals

In [62]:
NEW_FILE_NAME = "rentals_2026-07-02.csv"

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 64, Finished, Available, Finished, False)

In [63]:
new_file_bronze = (
    spark.table("bronze_rentals")
    .filter(
        F.lower(F.col("source_file"))
        .contains(NEW_FILE_NAME.lower())
    )
)

new_file_raw_count = new_file_bronze.count()

print(
    f"Raw rows in {NEW_FILE_NAME}: "
    f"{new_file_raw_count}"
)

assert new_file_raw_count > 0, (
    f"No Bronze rows found for {NEW_FILE_NAME}"
)

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 65, Finished, Available, Finished, False)

Raw rows in rentals_2026-07-02.csv: 5


In [64]:
new_file_valid, new_file_quarantine = (
    transform_bronze_rentals(new_file_bronze)
)

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 66, Finished, Available, Finished, False)

In [65]:
new_file_ids = (
    new_file_bronze
    .select("rental_id")
    .distinct()
)

existing_id_overlap = (
    new_file_ids
    .join(
        known_rental_ids,
        "rental_id",
        "inner"
    )
    .count()
)

print(
    "New-file IDs already known:",
    existing_id_overlap
)

assert existing_id_overlap == 0, (
    "The new-day file contains IDs already present "
    "in the existing Silver or quarantine tables."
)

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 67, Finished, Available, Finished, False)

New-file IDs already known: 0


In [66]:
expected_new_valid = (
    new_file_valid
    .select("rental_id")
    .distinct()
    .count()
)

expected_new_quarantined = (
    new_file_quarantine
    .select("rental_id")
    .distinct()
    .count()
)

print(
    "Expected valid rentals from new file:",
    expected_new_valid
)

print(
    "Expected quarantined rentals from new file:",
    expected_new_quarantined
)

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 68, Finished, Available, Finished, False)

Expected valid rentals from new file: 5
Expected quarantined rentals from new file: 0


#### 9. Rebuild Silver after adding the new file

In [67]:
new_file_run = build_silver_rentals()

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 69, Finished, Available, Finished, False)

{'silver_rentals': 952, 'silver_rentals_quarantine': 3}


In [68]:
after_silver_count = (
    spark.table("silver_rentals").count()
)

after_quarantine_count = (
    spark.table("silver_rentals_quarantine").count()
)

silver_increase = (
    after_silver_count - before_silver_count
)

quarantine_increase = (
    after_quarantine_count - before_quarantine_count
)

print(
    f"Silver before:       {before_silver_count}"
)

print(
    f"Silver after:        {after_silver_count}"
)

print(
    f"Silver increase:     {silver_increase}"
)

print(
    f"Expected increase:   {expected_new_valid}"
)

print(
    f"Quarantine increase: {quarantine_increase}"
)

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 70, Finished, Available, Finished, False)

Silver before:       947
Silver after:        952
Silver increase:     5
Expected increase:   5
Quarantine increase: 0


In [69]:
assert silver_increase == expected_new_valid, (
    f"Expected Silver to increase by "
    f"{expected_new_valid}, but it increased by "
    f"{silver_increase}"
)

assert (
    quarantine_increase
    == expected_new_quarantined
), (
    f"Expected quarantine to increase by "
    f"{expected_new_quarantined}, but it increased by "
    f"{quarantine_increase}"
)

assert (
    spark.table("silver_rentals")
    .groupBy("rental_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
) == 0

print(
    "Incremental-file test passed: only the new "
    "day's valid rentals were added."
)

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 71, Finished, Available, Finished, False)

Incremental-file test passed: only the new day's valid rentals were added.


In [70]:
third_run = build_silver_rentals()
fourth_run = build_silver_rentals()

assert third_run == fourth_run

print(
    "The Silver rebuild remains idempotent "
    "after adding the new daily file."
)

StatementMeta(, 18dbfe2e-7305-4ad7-8c72-9986a955139a, 72, Finished, Available, Finished, False)

{'silver_rentals': 952, 'silver_rentals_quarantine': 3}
{'silver_rentals': 952, 'silver_rentals_quarantine': 3}
The Silver rebuild remains idempotent after adding the new daily file.


#### The Silver rental build uses rental_id as its deterministic business key and reconstructs the valid and quarantine outputs from the complete Bronze snapshot on every run. A row_number window keeps only one latest source row per rental_id, after which the same null-safe quality rule sends impossible check-in records exclusively to quarantine. Both Delta tables are written with mode("overwrite") rather than append, so an identical rerun replaces the previous result instead of stacking another copy of it. Consequently, running the build twice produces identical counts, quarantined rentals cannot reappear in the valid table, and after a new daily file is landed, only its previously unseen valid rental IDs increase silver_rentals.